# Online Word Stroke Detection

**Task**: Given all strokes on a handwritten page, group them into words.

- **Input**: List of strokes (x, y coordinate arrays) on a page
- **Output**: Cluster assignment — which strokes belong to the same word
- **Evaluation**: Leave-one-page-out cross-validation (6 folds) on the xournalpp_htr benchmark dataset
- **Baseline**: Spatial proximity clustering (agglomerative, single linkage, bbox distance)

## 1. Data Loading

In [ ]:
import gzip
import json
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from bs4 import BeautifulSoup as bs
from bs4 import XMLParsedAsHTMLWarning
from huggingface_hub import snapshot_download
from sklearn.cluster import AgglomerativeClustering
from tabulate import tabulate

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

_TEXT_CLASSES = {"word", "digit", "mathematical_expression"}


@dataclass
class Stroke:
    x: np.ndarray
    y: np.ndarray


@dataclass
class PageData:
    page_id: str
    strokes: list[Stroke]
    word_groups: list[list[int]]

In [ ]:
def _parse_strokes_per_layer(path: Path) -> list[list[list[Stroke]]]:
    """Parse a .xopp/.xoj file and return strokes[page][layer]."""
    with gzip.open(path, "r") as f:
        content = f.read().decode("utf-8")

    soup = bs(content, "lxml")
    pages = []
    for page in soup.find_all("page"):
        layers = []
        for layer in page.find_all("layer"):
            layer_strokes = []
            for stroke_el in layer.find_all("stroke"):
                coords = np.fromstring(stroke_el.text, sep=" ").reshape(-1, 2).T
                layer_strokes.append(Stroke(x=coords[0], y=coords[1]))
            layers.append(layer_strokes)
        pages.append(layers)
    return pages


def load_benchmark_pages() -> list[PageData]:
    """Load all benchmark pages with their stroke-level word group annotations."""
    local_dir = Path(
        snapshot_download("PellelNitram/xournalpp_htr_benchmark", repo_type="dataset")
    )
    data_dir = local_dir / "data"
    xopp_files = sorted(data_dir.glob("*.xopp")) + sorted(data_dir.glob("*.xoj"))

    pages = []
    for xopp_path in xopp_files:
        gt_path = xopp_path.with_suffix("").with_suffix(".gt.json")
        if not gt_path.exists():
            continue

        strokes_per_layer = _parse_strokes_per_layer(xopp_path)

        with open(gt_path) as f:
            gt = json.load(f)

        page_annotations: dict[int, list[list[int]]] = {}
        for ann in gt["annotations"]:
            if ann["class"] not in _TEXT_CLASSES:
                continue
            page_idx = ann["page_index"]
            layer_idx = ann["layer_index"]

            offset = sum(
                len(strokes_per_layer[page_idx][li]) for li in range(layer_idx)
            )
            global_indices = [offset + si for si in ann["stroke_indices"]]
            page_annotations.setdefault(page_idx, []).append(global_indices)

        for page_idx, layer_list in enumerate(strokes_per_layer):
            flat_strokes = [s for layer in layer_list for s in layer]
            if not flat_strokes:
                continue
            word_groups = page_annotations.get(page_idx, [])
            page_id = f"{xopp_path.stem}_p{page_idx}"
            pages.append(
                PageData(
                    page_id=page_id,
                    strokes=flat_strokes,
                    word_groups=word_groups,
                )
            )

    return pages

In [ ]:
pages = load_benchmark_pages()

for p in pages:
    n_word_strokes = len({si for wg in p.word_groups for si in wg})
    print(
        f"{p.page_id}: {len(p.strokes)} strokes, "
        f"{len(p.word_groups)} words, "
        f"{n_word_strokes} word strokes"
    )

## 2. Visualize the Dataset

Plot each page with strokes colored by ground truth word group.

In [ ]:
def plot_page_gt(page: PageData, ax=None):
    """Plot strokes colored by ground truth word group."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(10, 6))

    cmap = plt.colormaps["tab20"]
    word_stroke_set = {si for wg in page.word_groups for si in wg}

    for gi, wg in enumerate(page.word_groups):
        color = cmap(gi % 20)
        for si in wg:
            s = page.strokes[si]
            ax.plot(s.x, -s.y, color=color, linewidth=1.5)

    for si, s in enumerate(page.strokes):
        if si not in word_stroke_set:
            ax.plot(s.x, -s.y, color="lightgray", linewidth=0.5)

    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(page.page_id, fontsize=10)


fig, axes = plt.subplots(3, 2, figsize=(16, 18))
for page, ax in zip(pages, axes.flat, strict=False):
    plot_page_gt(page, ax)
plt.suptitle("Ground Truth Word Groups", fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

## 3. Feature Extraction

Compute bounding boxes per stroke and pairwise min-edge distances between bounding boxes.

In [ ]:
def stroke_bbox(stroke: Stroke) -> tuple[float, float, float, float]:
    """Return (min_x, min_y, max_x, max_y) for a stroke."""
    return (
        float(stroke.x.min()),
        float(stroke.y.min()),
        float(stroke.x.max()),
        float(stroke.y.max()),
    )


def bbox_min_edge_distance(
    bbox_a: tuple[float, float, float, float],
    bbox_b: tuple[float, float, float, float],
) -> float:
    """Minimum distance between two axis-aligned bounding box edges. Returns 0 if overlapping."""
    ax0, ay0, ax1, ay1 = bbox_a
    bx0, by0, bx1, by1 = bbox_b
    dx = max(0.0, max(ax0 - bx1, bx0 - ax1))
    dy = max(0.0, max(ay0 - by1, by0 - ay1))
    return np.sqrt(dx * dx + dy * dy)


def compute_pairwise_bbox_distances(strokes: list[Stroke]) -> np.ndarray:
    """Compute NxN pairwise min-edge-distance matrix between stroke bounding boxes."""
    n = len(strokes)
    bboxes = [stroke_bbox(s) for s in strokes]
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = bbox_min_edge_distance(bboxes[i], bboxes[j])
            dist[i, j] = d
            dist[j, i] = d
    return dist

## 4. Evaluation Metrics

Match predicted stroke clusters to ground truth word groups using Jaccard index (IoU of stroke sets).

In [ ]:
@dataclass
class GroupingMetrics:
    precision: float
    recall: float
    f1: float
    stroke_accuracy: float
    n_gt_words: int
    n_pred_clusters: int


def evaluate_grouping(
    predicted_clusters: list[set[int]],
    gt_word_groups: list[list[int]],
    n_strokes: int,
    jaccard_threshold: float = 0.5,
) -> GroupingMetrics:
    """Evaluate predicted stroke clusters against ground truth word groups.

    A predicted cluster matches a GT word if their Jaccard index (intersection
    over union of stroke index sets) meets the threshold. Matching is greedy:
    best Jaccard first, one-to-one.
    """
    gt_sets = [set(wg) for wg in gt_word_groups]

    candidates = []
    for i, pred in enumerate(predicted_clusters):
        for j, gt in enumerate(gt_sets):
            intersection = len(pred & gt)
            union = len(pred | gt)
            if union > 0:
                jaccard = intersection / union
                if jaccard >= jaccard_threshold:
                    candidates.append((jaccard, i, j))

    candidates.sort(key=lambda x: x[0], reverse=True)

    matched_pred = set()
    matched_gt = set()
    for _, pi, gi in candidates:
        if pi not in matched_pred and gi not in matched_gt:
            matched_pred.add(pi)
            matched_gt.add(gi)

    n_matched = len(matched_pred)
    n_pred = len(predicted_clusters)
    n_gt = len(gt_sets)

    precision = n_matched / n_pred if n_pred > 0 else 0.0
    recall = n_matched / n_gt if n_gt > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    stroke_to_gt = {}
    for gi, gt in enumerate(gt_sets):
        for si in gt:
            stroke_to_gt[si] = gi

    stroke_to_pred = {}
    for pi, pred in enumerate(predicted_clusters):
        for si in pred:
            stroke_to_pred[si] = pi

    correct = 0
    total_word_strokes = len(stroke_to_gt)
    for si, gi in stroke_to_gt.items():
        if si in stroke_to_pred:
            pi = stroke_to_pred[si]
            pred_set = predicted_clusters[pi]
            gt_set = gt_sets[gi]
            intersection = len(pred_set & gt_set)
            union = len(pred_set | gt_set)
            if union > 0 and intersection / union >= jaccard_threshold:
                correct += 1

    stroke_accuracy = correct / total_word_strokes if total_word_strokes > 0 else 0.0

    return GroupingMetrics(
        precision=precision,
        recall=recall,
        f1=f1,
        stroke_accuracy=stroke_accuracy,
        n_gt_words=n_gt,
        n_pred_clusters=n_pred,
    )

## 5. Spatial Proximity Baseline

Cluster strokes using agglomerative clustering (single linkage) on bounding box distances. The distance threshold is tuned on training pages per CV fold.

In [ ]:
def cluster_strokes(strokes: list[Stroke], distance_threshold: float) -> list[set[int]]:
    """Cluster strokes using agglomerative clustering on bbox distance."""
    if len(strokes) == 0:
        return []
    if len(strokes) == 1:
        return [{0}]

    dist_matrix = compute_pairwise_bbox_distances(strokes)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="single",
        distance_threshold=distance_threshold,
    )
    labels = clustering.fit_predict(dist_matrix)

    clusters: dict[int, set[int]] = {}
    for i, label in enumerate(labels):
        clusters.setdefault(label, set()).add(i)

    return list(clusters.values())


def tune_threshold(
    train_pages: list[PageData],
    candidates: np.ndarray | None = None,
) -> float:
    """Find the distance threshold that maximises mean F1 on the training pages."""
    if candidates is None:
        candidates = np.arange(1.0, 100.0, 1.0)

    best_threshold = candidates[0]
    best_f1 = -1.0

    for threshold in candidates:
        f1s = []
        for page in train_pages:
            clusters = cluster_strokes(page.strokes, threshold)
            metrics = evaluate_grouping(clusters, page.word_groups, len(page.strokes))
            f1s.append(metrics.f1)
        mean_f1 = np.mean(f1s)
        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_threshold = threshold

    return float(best_threshold)

## 6. Leave-One-Page-Out Cross-Validation

In [ ]:
results = []
for i, test_page in enumerate(pages):
    train_pages = pages[:i] + pages[i + 1 :]
    threshold = tune_threshold(train_pages)
    clusters = cluster_strokes(test_page.strokes, threshold)
    metrics = evaluate_grouping(clusters, test_page.word_groups, len(test_page.strokes))
    results.append((test_page.page_id, threshold, metrics, clusters))
    print(
        f"Fold {i+1}/6: {test_page.page_id} — threshold={threshold:.1f}, F1={metrics.f1:.3f}"
    )

### Results Table

In [ ]:
headers = [
    "Page",
    "Threshold",
    "Precision",
    "Recall",
    "F1",
    "Stroke Acc",
    "GT Words",
    "Pred Clusters",
]
rows = []
for page_id, threshold, m, _ in results:
    rows.append(
        [
            page_id,
            f"{threshold:.1f}",
            f"{m.precision:.3f}",
            f"{m.recall:.3f}",
            f"{m.f1:.3f}",
            f"{m.stroke_accuracy:.3f}",
            m.n_gt_words,
            m.n_pred_clusters,
        ]
    )

precisions = [m.precision for _, _, m, _ in results]
recalls = [m.recall for _, _, m, _ in results]
f1s = [m.f1 for _, _, m, _ in results]
stroke_accs = [m.stroke_accuracy for _, _, m, _ in results]

rows.append(
    [
        "MEAN",
        "",
        f"{np.mean(precisions):.3f} ± {np.std(precisions):.3f}",
        f"{np.mean(recalls):.3f} ± {np.std(recalls):.3f}",
        f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
        f"{np.mean(stroke_accs):.3f} ± {np.std(stroke_accs):.3f}",
        "",
        "",
    ]
)

print(tabulate(rows, headers=headers, tablefmt="grid"))

## 7. Visualize Predictions vs Ground Truth

Side-by-side comparison for each page: GT word groups (left) vs predicted clusters (right).

In [ ]:
def plot_comparison(page: PageData, clusters: list[set[int]], title: str = ""):
    """Plot GT word groups vs predicted clusters side by side."""
    fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=(16, 8))
    cmap = plt.colormaps["tab20"]

    # Ground truth
    ax_gt.set_title("Ground Truth")
    word_stroke_set = {si for wg in page.word_groups for si in wg}
    for gi, wg in enumerate(page.word_groups):
        color = cmap(gi % 20)
        for si in wg:
            s = page.strokes[si]
            ax_gt.plot(s.x, -s.y, color=color, linewidth=1.5)
    for si, s in enumerate(page.strokes):
        if si not in word_stroke_set:
            ax_gt.plot(s.x, -s.y, color="lightgray", linewidth=0.5)
    ax_gt.set_aspect("equal")
    ax_gt.axis("off")

    # Predicted
    ax_pred.set_title("Predicted Clusters")
    clustered_strokes = {si for c in clusters for si in c}
    for ci, cluster in enumerate(clusters):
        color = cmap(ci % 20)
        for si in cluster:
            s = page.strokes[si]
            ax_pred.plot(s.x, -s.y, color=color, linewidth=1.5)
    for si, s in enumerate(page.strokes):
        if si not in clustered_strokes:
            ax_pred.plot(s.x, -s.y, color="lightgray", linewidth=0.5)
    ax_pred.set_aspect("equal")
    ax_pred.axis("off")

    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


for page_id, threshold, m, clusters in results:
    page = next(p for p in pages if p.page_id == page_id)
    plot_comparison(
        page, clusters, f"{page_id} (threshold={threshold:.1f}, F1={m.f1:.3f})"
    )

## 8. Threshold Sensitivity

Plot F1 as a function of distance threshold across all pages to understand the sensitivity.

In [ ]:
thresholds = np.arange(1.0, 80.0, 1.0)

fig, ax = plt.subplots(figsize=(12, 5))
cmap = plt.colormaps["tab10"]

all_f1s = np.zeros((len(pages), len(thresholds)))

for pi, page in enumerate(pages):
    f1s = []
    for threshold in thresholds:
        clusters = cluster_strokes(page.strokes, threshold)
        metrics = evaluate_grouping(clusters, page.word_groups, len(page.strokes))
        f1s.append(metrics.f1)
    all_f1s[pi] = f1s
    ax.plot(thresholds, f1s, color=cmap(pi % 10), alpha=0.5, label=page.page_id[-20:])

mean_f1 = all_f1s.mean(axis=0)
ax.plot(thresholds, mean_f1, color="black", linewidth=2, label="Mean F1")
ax.axvline(
    thresholds[np.argmax(mean_f1)],
    color="black",
    linestyle="--",
    alpha=0.5,
    label=f"Best mean threshold = {thresholds[np.argmax(mean_f1)]:.0f}",
)

ax.set_xlabel("Distance Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("F1 vs Distance Threshold (per page)")
ax.legend(fontsize=8, loc="lower left")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"Best mean F1 = {mean_f1.max():.3f} at threshold = {thresholds[np.argmax(mean_f1)]:.0f}"
)